# Hybrid Model — Colab Runner (Full-Image SR)

This notebook mounts your Google Drive and runs **test.py** against your **Hybrid** model in `Drive/hybrid/code` with the same folder layout as your local project.

**What it does:**
1) Installs deps (einops, scikit-image, opencv-python)
2) Mounts Drive
3) Points to `hybrid/code`, `hybrid/weights`, and `hybrid/data/TEST_data/{vi,ir}`
4) Imports your existing `test.py` and runs `test()` (no tiling by default)
5) Saves outputs to `hybrid/output` and previews a few images

> If you ever hit CUDA OOM on large images, there’s an **optional** cell later to switch to tiled SR (no quality drop). By default, we run full-image SR.


In [ ]:
# 1) Installs (lightweight; torch/torchvision come with Colab)
try:
    import einops, skimage, cv2
except Exception:
    !pip -q install einops scikit-image opencv-python
print('Deps ready.')


Deps ready.


In [1]:
# 3) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
# 4) Configure your paths here
# Change DRIVE_ROOT if your "hybrid" folder lives somewhere else in Drive.
DRIVE_ROOT = '/content/drive/MyDrive/Hybrid'  # <-- adjust if needed

import os
CODE_DIR   = f"/content/drive/MyDrive/Hybrid/Code"
WEIGHTS_DIR= f"/content/drive/MyDrive/Hybrid/weights"
VI_DIR     = f"/content/drive/MyDrive/Hybrid/data/TEST_data/vi"
IR_DIR     = f"/content/drive/MyDrive/Hybrid/data/TEST_data/ir"
OUT_DIR    = f"/content/drive/MyDrive/Hybrid/output/ablation_e_output/"

# Weight files (change names if yours differ)
CKPT_LEFUSE = f"/content/drive/MyDrive/Hybrid/weights/L2025.pth"
CKPT_UNET   = f"/content/drive/MyDrive/Hybrid/weights/sr_unet_aug_enhanced.pth"

for p in [CODE_DIR, WEIGHTS_DIR, VI_DIR, IR_DIR]:
    assert os.path.exists(p), f"Missing path: {p}"
os.makedirs(OUT_DIR, exist_ok=True)
print('Paths OK')


Paths OK


In [3]:
# 5) Add code folder to sys.path and list key files
import sys, glob
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print('sys.path[0]:', sys.path[0])
print('Code files:\n', '\n'.join(sorted([os.path.basename(p) for p in glob.glob(CODE_DIR+'/*.py')])))


sys.path[0]: /content/drive/MyDrive/Hybrid/Code
Code files:
 Copy of hybrid_model.py
hybrid_model.py
hybrid_model_ablation_a.py
hybrid_model_ablation_b.py
hybrid_model_ablation_c.py
hybrid_model_ablation_d.py
img_utils.py
test.py
test_ablation_a.py
test_enhance.py
test_in_chunk.py
test_wrong.py
utils.py
vgg.py


In [4]:
# 6) Import modules and basic sanity checks
import importlib
hm = importlib.import_module('hybrid_model')
test_mod = importlib.import_module('test')  # your test.py

print('Imported hybrid_model.py and test.py')
print('hybrid_model exposes:', [n for n in dir(hm) if n.lower() in ('hybridmodel','lefuse','srunet','enhancedsrunet','selfensemblewrapper')])
print('test module has test():', hasattr(test_mod, 'test'))


Imported hybrid_model.py and test.py
hybrid_model exposes: ['EnhancedSRUnet', 'HybridModel', 'LEFuse', 'SRUnet']
test module has test(): True


In [ ]:
# 7) Run FULL-IMAGE super-resolution (no tiling) via your test.py
from test import test as run_test
run_test(CKPT_LEFUSE, CKPT_UNET, VI_DIR, IR_DIR, OUT_DIR)
print('Done running test(). Outputs saved into:', OUT_DIR)


### Optional: OOM-safe mode (tiled SR)
If you see **CUDA out of memory** for very large images, run the cell below **instead of** the "Run FULL-IMAGE" cell. It switches to a tiled SR path inside the notebook while keeping the same model/weights and visual quality.

In [ ]:
# 8) OPTIONAL — OOM-safe tiled SR (accelerated)
import os, importlib, numpy as np, torch
from skimage.io import imsave
from img_utils import image_read_cv2, RGB2YCrCb, YCbCr2RGB

# ---- speed knobs ----
USE_AMP = True                 # mixed precision on GPU (faster, lower VRAM)
BASE_TILE = 384                # starting guess; will auto-shrink if VRAM is tight
OVERLAP = 32                   # 16 is faster but a bit more seam-prone
HANN_BLEND = True              # keep True for seamless borders
PREFERS_ENSEMBLE = True        # if HybridModel has .ensemble, use it; else unet

# cuDNN speedups
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

def _hann_window(h, w, device, dtype):
    wy = torch.hann_window(h, periodic=False, device=device, dtype=dtype).view(1,1,h,1)
    wx = torch.hann_window(w, periodic=False, device=device, dtype=dtype).view(1,1,1,w)
    return (wy*wx).clamp(min=1e-6)

def _adaptive_tile(img_hw, base_tile=BASE_TILE, overlap=OVERLAP, scale=4):
    """
    Pick a tile size as large as we reasonably can based on free VRAM.
    Safe heuristic: shrink until predicted tile output fits in free_mem/4.
    """
    H, W = img_hw
    tile = min(base_tile, H, W)
    if torch.cuda.is_available():
        free_mem, total_mem = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024**3)
        # very rough budget per tile (in GB), leave headroom for model/activations
        budget_gb = max(0.6, min(3.0, free_gb * 0.35))
        # predicted out tile size (bytes) ~ 1*3*(tile*scale)^2*2 bytes (fp16) * factor
        factor = 6.0  # overhead fudge (model activations etc.)
        while tile > 128:
            out_pix = (tile*scale)*(tile*scale)
            predicted_gb = (3*out_pix*2) / (1024**3) * factor  # fp16 bytes * factor
            if predicted_gb <= budget_gb:
                break
            tile -= 32
    return max(128, tile)

@torch.inference_mode()
def _sr_infer_tiled(sr_callable, img, scale=4, tile=None, overlap=OVERLAP, use_hann=HANN_BLEND, use_amp=USE_AMP):
    """
    Tiled SR inference. img: [1,3,H,W] in [0,1] on device.
    """
    _, _, H, W = img.shape
    device = img.device
    dtype = torch.float16 if (use_amp and torch.cuda.is_available()) else img.dtype
    tile = _adaptive_tile((H, W), BASE_TILE if tile is None else tile, overlap, scale)

    outH, outW = H * scale, W * scale
    out_acc = torch.zeros(1, 3, outH, outW, device=device, dtype=dtype)
    out_wgt = torch.zeros_like(out_acc)

    stride = tile - overlap
    ys = list(range(0, H, stride))
    xs = list(range(0, W, stride))
    if ys[-1] != H - 1: ys[-1] = max(0, H - tile)
    if xs[-1] != W - 1: xs[-1] = max(0, W - tile)

    amp_ctx = torch.autocast if (use_amp and torch.cuda.is_available()) else torch.cpu.amp.autocast
    with amp_ctx(device_type='cuda', enabled=use_amp):
        for y0 in ys:
            for x0 in xs:
                y1 = min(y0 + tile, H)
                x1 = min(x0 + tile, W)
                inp = img[:, :, y0:y1, x0:x1].to(dtype, copy=False)
                out_tile = sr_callable(inp)  # [1,3,(y1-y0)*s,(x1-x0)*s]
                oy0, ox0 = y0 * scale, x0 * scale
                oy1, ox1 = y1 * scale, x1 * scale
                if use_hann:
                    win = _hann_window(out_tile.shape[-2], out_tile.shape[-1], out_tile.device, out_tile.dtype)
                    out_acc[:, :, oy0:oy1, ox0:ox1] += out_tile * win
                    out_wgt[:, :, oy0:oy1, ox0:ox1] += win
                else:
                    out_acc[:, :, oy0:oy1, ox0:ox1] += out_tile
                    out_wgt[:, :, oy0:oy1, ox0:ox1] += 1.0
                del inp, out_tile

    out = out_acc / (out_wgt + 1e-8)
    # Return in float32 for consistent downstream stats/saving
    return out.clamp_(0, 1).to(torch.float32)

# --- load model the same way you do ---
hm = importlib.import_module('hybrid_model')
HybridModel = hm.HybridModel; LEFuse = hm.LEFuse
UnetClass = getattr(hm, 'SRUnet', None) or getattr(hm, 'EnhancedSRUnet', None)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    hybrid_model = HybridModel().to(device)
    lefuse = hybrid_model.lefuse; unet = hybrid_model.unet
    sr_core = getattr(hybrid_model, 'ensemble', unet) if PREFERS_ENSEMBLE else unet
except TypeError:
    lefuse = LEFuse().to(device)
    unet = UnetClass(in_channels=3, out_channels=3, scale=4).to(device)
    hybrid_model = HybridModel(lefuse, unet).to(device)
    sr_core = unet  # no ensemble available in old API

# weights
sd_l = torch.load(CKPT_LEFUSE, map_location=device)
if isinstance(sd_l, dict) and 'model' in sd_l: sd_l = sd_l['model']
lefuse_sd = {}
for k, v in sd_l.items():
    if k.startswith('content.fuser.'): lefuse_sd[k.replace('content.fuser.', 'fuser.')] = v
    elif k.startswith('fuser.'): lefuse_sd[k] = v
lefuse.load_state_dict(lefuse_sd, strict=False)

sd_u = torch.load(CKPT_UNET, map_location=device)
tgt = unet.state_dict(); comp = {k:v for k,v in sd_u.items() if k in tgt and v.shape == tgt[k].shape}
tgt.update(comp); unet.load_state_dict(tgt, strict=False)

hybrid_model.eval(); lefuse.eval(); unet.eval()

# --- run ---
files = sorted([f for f in os.listdir(VI_DIR) if f.lower().endswith(('.png','.jpg','.jpeg'))])
with torch.inference_mode():
    for img_name in files:
        # I/O
        vi_np = image_read_cv2(os.path.join(VI_DIR, img_name), mode='RGB')[None, ...] / 255.0
        vi = torch.from_numpy(vi_np.transpose(0,3,1,2)).to(device, dtype=torch.float32)
        ir_np = image_read_cv2(os.path.join(IR_DIR, img_name), mode='GRAY')[None, None, ...] / 255.0
        ir = torch.from_numpy(ir_np).to(device, dtype=torch.float32)

        # fuse on luminance
        y, cr, cb = RGB2YCrCb(vi)
        fused_y = lefuse(y, ir)
        sr_in = YCbCr2RGB(fused_y, cb, cr)  # [1,3,H,W] float32

        # fast tiled SR
        out = _sr_infer_tiled(sr_core, sr_in, scale=4, tile=None, overlap=OVERLAP,
                              use_hann=HANN_BLEND, use_amp=USE_AMP)

        # save
        os.makedirs(f"{OUT_DIR}/debug", exist_ok=True)
        os.makedirs(f"{OUT_DIR}/intermediate", exist_ok=True)
        sr_in_np = (sr_in.detach().cpu().numpy().squeeze(0).transpose(1,2,0)*255.0).clip(0,255).astype('uint8')
        imsave(f"{OUT_DIR}/debug/{os.path.splitext(img_name)[0]}_sr_input.png", sr_in_np)
        out_np = (out.detach().cpu().numpy().squeeze(0).transpose(1,2,0)*255.0).clip(0,255).astype('uint8')
        imsave(f"{OUT_DIR}/{os.path.splitext(img_name)[0]}_fused_hr.png", out_np)

print('Tiled SR (accelerated) complete. Outputs in', OUT_DIR)


In [ ]:
# 8) OPTIONAL — OOM-safe tiled SR (accelerated) + Ablation metrics + FLOPs/Params/Latency + Excel
# Uses YOUR already-defined paths/ckpts in this notebook:
#   CKPT_LEFUSE, CKPT_UNET, VI_DIR, IR_DIR, OUT_DIR
# Also supports running multiple ablation modules in one go (see MODULES below).
# Requires: pandas, scikit-image; optional: openpyxl (Excel), thop (FLOPs).

import os, time, math, importlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from skimage.io import imsave
from skimage.metrics import structural_similarity as ssim
from skimage.filters import sobel_h, sobel_v
from skimage.transform import resize as sk_resize
from img_utils import image_read_cv2, RGB2YCrCb, YCbCr2RGB

# ------------------------------------------------------------------------------------
# Speed/VRAM knobs
USE_AMP = True                 # mixed precision on GPU (faster, lower VRAM)
BASE_TILE = 384                # starting tile; will auto-shrink if VRAM is tight
OVERLAP = 32                   # 16 is a bit faster but can show seams
HANN_BLEND = True              # smooth tile borders
PREFERS_ENSEMBLE = True        # if HybridModel has .ensemble, use it; else UNet

# cuDNN speedups
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# ------------------------------------------------------------------------------------
# WHAT TO RUN: add/remove modules here. Baseline + ablations are supported.
# (Comment out any you don't have.)
MODULES = [
    ("BASE", "hybrid_model"),
    ("A",    "hybrid_model_ablation_a"),
    ("B",    "hybrid_model_ablation_b"),
    ("C",    "hybrid_model_ablation_c"),
    ("D",    "hybrid_model_ablation_d"),
    ("E",    "hybrid_model_ablation_e"),
]

# ------------------------------------------------------------------------------------
# Sanity on your pre-defined notebook variables
for _name in ["CKPT_LEFUSE", "CKPT_UNET", "VI_DIR", "IR_DIR", "OUT_DIR"]:
    if _name not in globals():
        raise RuntimeError(f"Notebook variable `{_name}` is not defined above this cell.")
for _p in [VI_DIR, IR_DIR, OUT_DIR]:
    os.makedirs(_p, exist_ok=True)

# ------------------------------------------------------------------------------------
# Utility functions
def _hann_window(h, w, device, dtype):
    wy = torch.hann_window(h, periodic=False, device=device, dtype=dtype).view(1,1,h,1)
    wx = torch.hann_window(w, periodic=False, device=device, dtype=dtype).view(1,1,1,w)
    return (wy * wx).clamp(min=1e-6)

def _adaptive_tile(img_hw, base_tile=BASE_TILE, overlap=OVERLAP, scale=4):
    H, W = img_hw
    tile = min(base_tile, H, W)
    if torch.cuda.is_available():
        free_mem, _ = torch.cuda.mem_get_info()
        free_gb = free_mem / (1024**3)
        budget_gb = max(0.6, min(3.0, free_gb * 0.35))  # headroom
        factor = 6.0  # rough overhead fudge
        while tile > 128:
            out_pix = (tile*scale) * (tile*scale)
            predicted_gb = (3*out_pix*2) / (1024**3) * factor  # fp16 bytes * factor
            if predicted_gb <= budget_gb:
                break
            tile -= 32
    return max(128, tile)

@torch.inference_mode()
def _sr_infer_tiled(sr_callable, img, scale=4, tile=None, overlap=OVERLAP, use_hann=HANN_BLEND, use_amp=USE_AMP):
    """
    Tiled SR inference. img: [1,3,H,W] in [0,1] on device. Returns [1,3,H*scale,W*scale] float32 in [0,1].
    """
    _, _, H, W = img.shape
    device = img.device
    dtype = torch.float16 if (use_amp and torch.cuda.is_available()) else img.dtype
    tile = _adaptive_tile((H, W), BASE_TILE if tile is None else tile, overlap, scale)

    outH, outW = H * scale, W * scale
    out_acc = torch.zeros(1, 3, outH, outW, device=device, dtype=dtype)
    out_wgt = torch.zeros_like(out_acc)

    stride = tile - overlap
    ys = list(range(0, H, stride))
    xs = list(range(0, W, stride))
    if ys[-1] != H - 1: ys[-1] = max(0, H - tile)
    if xs[-1] != W - 1: xs[-1] = max(0, W - tile)

    amp_ctx = torch.autocast if (use_amp and torch.cuda.is_available()) else torch.cpu.amp.autocast
    with amp_ctx(device_type='cuda', enabled=use_amp):
        for y0 in ys:
            for x0 in xs:
                y1 = min(y0 + tile, H)
                x1 = min(x0 + tile, W)
                inp = img[:, :, y0:y1, x0:x1].to(dtype, copy=False)
                out_tile = sr_callable(inp)  # -> [1,3,(y1-y0)*s,(x1-x0)*s]
                oy0, ox0 = y0 * scale, x0 * scale
                oy1, ox1 = y1 * scale, x1 * scale
                if use_hann:
                    win = _hann_window(out_tile.shape[-2], out_tile.shape[-1], out_tile.device, out_tile.dtype)
                    out_acc[:, :, oy0:oy1, ox0:ox1] += out_tile * win
                    out_wgt[:, :, oy0:oy1, ox0:ox1] += win
                else:
                    out_acc[:, :, oy0:oy1, ox0:ox1] += out_tile
                    out_wgt[:, :, oy0:oy1, ox0:ox1] += 1.0
                del inp, out_tile

    out = out_acc / (out_wgt + 1e-8)
    return out.clamp_(0, 1).to(torch.float32)

def _tensor_to_uint8_img(t: torch.Tensor):
    arr = t.detach().cpu().numpy().squeeze(0)
    if arr.ndim == 3:
        arr = np.transpose(arr, (1, 2, 0))
    return (np.clip(arr, 0, 1) * 255.0).astype(np.uint8)

def _to_gray_uint8(arr):
    if arr.ndim == 3 and arr.shape[2] == 3:
        arr = 0.299*arr[...,0] + 0.587*arr[...,1] + 0.114*arr[...,2]
    arr = np.clip(arr, 0, 1)
    return (arr * 255.0).astype(np.uint8)

def _entropy(gray_uint8):
    hist = np.bincount(gray_uint8.ravel(), minlength=256).astype(np.float64)
    p = hist / (gray_uint8.size + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _spatial_frequency(gray_float):
    rf = np.diff(gray_float, axis=0); cf = np.diff(gray_float, axis=1)
    RF = np.sqrt(np.mean(rf * rf)); CF = np.sqrt(np.mean(cf * cf))
    return float(np.sqrt(RF*RF + CF*CF))

def _avg_gradient(gray_float):
    gx = sobel_h(gray_float); gy = sobel_v(gray_float)
    return float(np.mean(np.hypot(gx, gy)))

def _mutual_information(gray_a_uint8, gray_b_uint8, bins=256):
    jh, _, _ = np.histogram2d(gray_a_uint8.ravel(), gray_b_uint8.ravel(), bins=bins, range=[[0,255],[0,255]])
    pxy = jh / (jh.sum() + 1e-12); px = pxy.sum(axis=1, keepdims=True); py = pxy.sum(axis=0, keepdims=True)
    pxpy = px @ py; nz = pxy > 0
    return float((pxy[nz] * np.log2(pxy[nz] / (pxpy[nz] + 1e-12))).sum())

def _ssim_safe(a_float, b_float):
    if a_float.shape != b_float.shape:
        b_float = sk_resize(b_float, a_float.shape, order=1, anti_aliasing=True, preserve_range=True)
    return float(ssim(a_float, b_float, data_range=1.0))

def _ensure_dirs(path):
    os.makedirs(path, exist_ok=True)
    os.makedirs(os.path.join(path, 'debug'), exist_ok=True)
    os.makedirs(os.path.join(path, 'intermediate'), exist_ok=True)

def _load_lefuse_weights(lefuse, ckpt, device):
    sd = torch.load(ckpt, map_location=device)
    if isinstance(sd, dict) and "model" in sd: sd = sd["model"]
    filt = {}
    for k, v in sd.items():
        if k.startswith('content.fuser.'): filt[k.replace('content.fuser.', 'fuser.')] = v
        elif k.startswith('fuser.'):       filt[k] = v
    if not filt: raise RuntimeError("LEFuse ckpt has no fuser.* or content.fuser.* keys")
    lefuse.load_state_dict(filt, strict=False)

def _load_unet_weights(unet, ckpt, device):
    src = torch.load(ckpt, map_location=device)
    tgt = unet.state_dict()
    compat = {k: v for k, v in src.items() if k in tgt and v.shape == tgt[k].shape}
    tgt.update(compat)
    unet.load_state_dict(tgt, strict=False)

def _profile_flops_params(hybrid_model, lefuse, unet, device, H=256, W=256, scale=4):
    """
    Returns dict with params (M) and FLOPs (G) for (LEFuse, SRNet, HybridModel). Uses THOP if available.
    """
    out = dict(
        params_M_total=np.nan, params_M_fuser=np.nan, params_M_sr=np.nan,
        flops_G_total=np.nan, flops_G_fuser=np.nan, flops_G_sr=np.nan
    )
    # Params
    try:
        p_total = sum(p.numel() for p in hybrid_model.parameters()) if hasattr(hybrid_model, 'parameters') else 0
        p_fuse  = sum(p.numel() for p in lefuse.parameters())
        p_sr    = sum(p.numel() for p in unet.parameters())
        out.update(
            params_M_total=round(p_total/1e6, 3),
            params_M_fuser=round(p_fuse/1e6, 3),
            params_M_sr=round(p_sr/1e6, 3),
        )
    except Exception:
        pass

    # FLOPs
    try:
        from thop import profile
        rgb = torch.randn(1, 3, H, W, device=device)
        ir  = torch.randn(1, 1, H, W, device=device)

        # Try full model
        flops_total = None
        try:
            hybrid_model.eval()
            flops_total, _ = profile(hybrid_model, inputs=(rgb, ir), verbose=False)
        except Exception:
            with torch.no_grad():
                y, cr, cb = RGB2YCrCb(rgb)
                flops_fuser, _ = profile(lefuse, inputs=(y, ir), verbose=False)
                fused_y = lefuse(y, ir)
                fused_rgb_lr = YCbCr2RGB(fused_y, cb, cr)
                flops_sr, _ = profile(unet, inputs=(fused_rgb_lr,), verbose=False)
                flops_total = flops_fuser + flops_sr
                out["flops_G_fuser"] = round(flops_fuser/1e9, 3)
                out["flops_G_sr"]    = round(flops_sr/1e9, 3)

        if flops_total is not None:
            out["flops_G_total"] = round(flops_total/1e9, 3)

        # Try to fill component FLOPs if missing
        if (isinstance(out["flops_G_fuser"], float) and math.isnan(out["flops_G_fuser"])) or \
           (isinstance(out["flops_G_sr"], float) and math.isnan(out["flops_G_sr"])):
            with torch.no_grad():
                y, cr, cb = RGB2YCrCb(rgb)
                f_fuse, _ = profile(lefuse, inputs=(y, ir), verbose=False)
                fused_y = lefuse(y, ir)
                fused_rgb_lr = YCbCr2RGB(fused_y, cb, cr)
                f_sr, _ = profile(unet, inputs=(fused_rgb_lr,), verbose=False)
            out["flops_G_fuser"] = round(f_fuse/1e9, 3)
            out["flops_G_sr"]    = round(f_sr/1e9, 3)

    except Exception:
        # thop not installed or failed; keep NaNs
        pass

    return out

# ------------------------------------------------------------------------------------
# Gather files and ensure matching names
exts = ('.png', '.jpg', '.jpeg')
vi_files = sorted([f for f in os.listdir(VI_DIR) if f.lower().endswith(exts)])
ir_files = sorted([f for f in os.listdir(IR_DIR) if f.lower().endswith(exts)])
if vi_files != ir_files:
    raise ValueError("Mismatched filenames between VI_DIR and IR_DIR")
filenames = vi_files

device = 'cuda' if torch.cuda.is_available() else 'cpu'
all_rows, summary_rows = [], []

# ------------------------------------------------------------------------------------
# Run each module with tiled SR + metrics + FLOPs/Params + latency
for label, modname in MODULES:
    # Import module; skip if absent
    try:
        hm = importlib.import_module(modname)
        hm = importlib.reload(hm)
    except Exception as e:
        print(f"[{label}] SKIP: cannot import {modname} -> {e}")
        continue

    # Build model (support both APIs)
    try:
        try:
            hybrid_model = hm.HybridModel().to(device).eval()
            lefuse = getattr(hybrid_model, "lefuse", None)
            unet   = getattr(hybrid_model, "unet", None)
            if lefuse is None or unet is None:
                raise RuntimeError("HybridModel() did not expose .lefuse/.unet")
            sr_core = getattr(hybrid_model, 'ensemble', unet) if PREFERS_ENSEMBLE else unet
        except TypeError:
            # Old signature: (lefuse, unet)
            lefuse = hm.LEFuse().to(device)
            UnetClass = getattr(hm, 'SRUnet', None) or getattr(hm, 'EnhancedSRUnet', None)
            if UnetClass is None:
                raise RuntimeError(f"{modname} defines neither SRUnet nor EnhancedSRUnet")
            unet = UnetClass(in_channels=3, out_channels=3, scale=4).to(device)
            hybrid_model = hm.HybridModel(lefuse, unet).to(device).eval()
            sr_core = unet
    except Exception as e:
        print(f"[{label}] SKIP: build failed -> {e}")
        continue

    # Load weights (lenient)
    try:
        _load_lefuse_weights(lefuse, CKPT_LEFUSE, device)
    except Exception as e:
        print(f"[{label}] WARN: LEFuse weights -> {e}")
    try:
        _load_unet_weights(unet, CKPT_UNET, device)
    except Exception as e:
        print(f"[{label}] WARN: UNet weights -> {e}")

    # FLOPs/Params (optional, needs thop)
    flops_info = _profile_flops_params(hybrid_model, lefuse, unet, device, H=256, W=256, scale=4)

    # Prepare output dirs
    variant_out = os.path.join(OUT_DIR, f"variant_{label}")
    _ensure_dirs(variant_out)

    # Per-variant aggregation
    latency_ms = []
    per_variant = []

    with torch.inference_mode():
        for img_name in filenames:
            try:
                # Read VI/IR
                vi_np = image_read_cv2(os.path.join(VI_DIR, img_name), mode='RGB')[None, ...] / 255.0
                vi = torch.from_numpy(vi_np.transpose(0,3,1,2)).to(device, dtype=torch.float32)
                ir_np = image_read_cv2(os.path.join(IR_DIR, img_name), mode='GRAY')[None, None, ...] / 255.0
                ir = torch.from_numpy(ir_np).to(device, dtype=torch.float32)

                # Forward: fuse on Y, then tiled SR on RGB
                t0 = time.time()
                y, cr, cb = RGB2YCrCb(vi)
                fused_y = lefuse(y, ir)                       # [1,1,H,W]
                sr_in   = YCbCr2RGB(fused_y, cb, cr)          # [1,3,H,W]
                sr_callable = (lambda x: hybrid_model.ensemble(x)) if (PREFERS_ENSEMBLE and hasattr(hybrid_model, "ensemble")) else (lambda x: unet(x))
                out = _sr_infer_tiled(sr_callable, sr_in, scale=4, tile=None, overlap=OVERLAP, use_hann=HANN_BLEND, use_amp=USE_AMP)
                if device.startswith('cuda'): torch.cuda.synchronize()
                latency_ms.append((time.time() - t0) * 1000.0)

                # Save images
                os.makedirs(f"{variant_out}/debug", exist_ok=True)
                os.makedirs(f"{variant_out}/intermediate", exist_ok=True)
                sr_in_u8 = (sr_in.detach().cpu().numpy().squeeze(0).transpose(1,2,0)*255.0).clip(0,255).astype('uint8')
                out_u8   = (out.detach().cpu().numpy().squeeze(0).transpose(1,2,0)*255.0).clip(0,255).astype('uint8')
                imsave(f"{variant_out}/debug/{os.path.splitext(img_name)[0]}_sr_input.png", sr_in_u8)
                out_hr_path = f"{variant_out}/{os.path.splitext(img_name)[0]}_fused_hr.png"
                imsave(out_hr_path, out_u8)

                # ---------------- Metrics ----------------
                # Fusion (LR) luminance metrics (no-ref)
                fy = np.clip(fused_y.detach().cpu().numpy().squeeze().astype(np.float32), 0, 1)
                vy = np.clip(y.detach().cpu().numpy().squeeze().astype(np.float32), 0, 1)
                irn = np.clip(ir.detach().cpu().numpy().squeeze().astype(np.float32), 0, 1)

                ENT_Y = _entropy(_to_gray_uint8(fy))
                SF_Y  = _spatial_frequency(fy)
                AG_Y  = _avg_gradient(fy)

                SSIM_F_VI_Y = _ssim_safe(fy, vy)
                SSIM_F_IR   = _ssim_safe(fy, irn)
                MI_F_VI_Y   = _mutual_information(_to_gray_uint8(fy), _to_gray_uint8(vy))
                MI_F_IR     = _mutual_information(_to_gray_uint8(fy), _to_gray_uint8(irn))

                # Final HR metrics (no-ref, on luminance)
                fhr = np.clip(out.detach().cpu().numpy().squeeze(0).transpose(1,2,0).astype(np.float32), 0, 1)
                fhr_y = 0.299*fhr[...,0] + 0.587*fhr[...,1] + 0.114*fhr[...,2]
                ENT_HR = _entropy(_to_gray_uint8(fhr))
                SF_HR  = _spatial_frequency(fhr_y)
                AG_HR  = _avg_gradient(fhr_y)

                per_variant.append(dict(
                    variant=label, module=modname, image=img_name,
                    ENT_Y=ENT_Y, SF_Y=SF_Y, AG_Y=AG_Y,
                    SSIM_F_VI_Y=SSIM_F_VI_Y, SSIM_F_IR=SSIM_F_IR,
                    MI_F_VI_Y=MI_F_VI_Y, MI_F_IR=MI_F_IR,
                    ENT_HR=ENT_HR, SF_HR=SF_HR, AG_HR=AG_HR,
                    out_hr=out_hr_path
                ))
                print(f"[{label}] {img_name}: OK ({latency_ms[-1]:.1f} ms)")

            except Exception as e:
                print(f"[{label}] {img_name}: ERROR -> {e}")

    # Per-variant summary
    if per_variant:
        dfv = pd.DataFrame(per_variant)
        means = dfv.drop(columns=["variant","module","image","out_hr"]).mean(numeric_only=True).to_dict()
        avg_ms = float(np.mean(latency_ms)) if latency_ms else np.nan
        ips = 1000.0/avg_ms if (avg_ms and not math.isnan(avg_ms) and avg_ms>0) else np.nan
        means.update(
            variant=label, module=modname, num_images=len(dfv),
            avg_latency_ms=round(avg_ms, 2), throughput_img_per_s=round(ips, 2),
            params_M_total=flops_info.get("params_M_total", np.nan),
            params_M_fuser=flops_info.get("params_M_fuser", np.nan),
            params_M_sr=flops_info.get("params_M_sr", np.nan),
            flops_G_total=flops_info.get("flops_G_total", np.nan),
            flops_G_fuser=flops_info.get("flops_G_fuser", np.nan),
            flops_G_sr=flops_info.get("flops_G_sr", np.nan),
        )
        summary_rows.append(means)
        # Append per-image to global
        all_rows.extend(per_variant)
        print(f"[{label}] Summary -> {means}")
    else:
        print(f"[{label}] No images processed.")

# ------------------------------------------------------------------------------------
# Save results (Excel preferred, CSV fallback)
if not all_rows:
    raise RuntimeError("No results produced. Check imports, inputs, and that images exist in VI_DIR/IR_DIR.")

df_all = pd.DataFrame(all_rows)
df_sum = pd.DataFrame(summary_rows)

excel_path = os.path.join(OUT_DIR, "ablation_metrics.xlsx")
try:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
        df_all.to_excel(w, sheet_name="per_image", index=False)
        df_sum.to_excel(w, sheet_name="summary_by_variant", index=False)
    print(f"Saved Excel: {excel_path}")
except Exception as e:
    csv_all = os.path.join(OUT_DIR, "ablation_metrics_per_image.csv")
    csv_sum = os.path.join(OUT_DIR, "ablation_metrics_summary.csv")
    df_all.to_csv(csv_all, index=False)
    df_sum.to_csv(csv_sum, index=False)
    print(f"Excel write failed ({e}); saved CSVs instead:\n- {csv_all}\n- {csv_sum}")


In [ ]:
# 9) Preview outputs with NO-REFERENCE metrics (BRISQUE, NIQE, Sharpness)
import os, glob, numpy as np, matplotlib.pyplot as plt

# Install pyiqa if needed
try:
    import pyiqa
except ImportError:
    !pip -q install pyiqa
    import pyiqa

import torch
import cv2
from skimage.io import imread

# Init models once (on CPU to avoid VRAM use)
brisque_model = pyiqa.create_metric('brisque', device='cpu')
niqe_model    = pyiqa.create_metric('niqe', device='cpu')

def lap_var(im):
    """Laplacian variance as a simple sharpness proxy (higher is sharper)."""
    if im.ndim == 3:
        gray = cv2.cvtColor(im, cv2.COLOR_RGB2GRAY)
    else:
        gray = im
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

outs = sorted(glob.glob(OUT_DIR+'/*_fused_hr.png'))[:4]

for p in outs:
    img = imread(p)
    if img.dtype != np.uint8:
        img_u8 = (np.clip(img, 0, 1) * 255).astype(np.uint8) if img.max() <= 1.0 else img.astype(np.uint8)
    else:
        img_u8 = img

    # Convert to tensor [1,3,H,W], normalized to 0..1 for pyiqa
    img_tensor = torch.from_numpy(img_u8).permute(2,0,1).unsqueeze(0).float() / 255.0

    # Compute metrics
    try:
        bq = float(brisque_model(img_tensor))
    except Exception as e:
        bq = None
    try:
        nq = float(niqe_model(img_tensor))
    except Exception as e:
        nq = None
    lv = lap_var(img_u8)

    title = os.path.basename(p)
    parts = []
    if bq is not None: parts.append(f"BRISQUE: {bq:.2f}")
    if nq is not None: parts.append(f"NIQE: {nq:.2f}")
    parts.append(f"LapVar: {lv:.0f}")
    title += "  |  " + "   ".join(parts)

    plt.figure(figsize=(6,6))
    plt.title(title)
    plt.imshow(img)
    plt.axis('off')

plt.show()
print('Previewed', len(outs), 'image(s) with BRISQUE/NIQE/Laplacian scores (no-reference).')


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 6.7 MB/s eta 0:00:00


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.3/172.3 kB 13.4 MB/s eta 0:00:00


In [ ]:
# Compute EN, AG, SD, SF on debug images + PSNR/SSIM (vs bicubic) on final outputs, save to Excel
import os, glob, math
import numpy as np
import pandas as pd
import cv2
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ---- config ----
OUTPUT_XLS = os.path.join(OUT_DIR, "M3FD_50-images_metrics.xlsx")   # Excel file path to save
HR_GLOB    = os.path.join(OUT_DIR, "*_fused_hr.png") # final outputs
DBG_DIR    = os.path.join(OUT_DIR, "debug")          # where *_sr_input.png lives

# ---- helpers ----
def to_u8(img):
    if img.dtype == np.uint8:
        return img
    if img.max() <= 1.0:
        return (np.clip(img, 0, 1) * 255.0).astype(np.uint8)
    return img.astype(np.uint8)

def read_rgb(path):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def read_gray(path):
    rgb = read_rgb(path)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    return gray

def entropy_EN(gray_u8):
    hist = cv2.calcHist([gray_u8],[0],None,[256],[0,256]).ravel()
    p = hist / (hist.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def average_gradient_AG(gray_f):
    # finite differences
    dx = gray_f[1:, :] - gray_f[:-1, :]
    dy = gray_f[:, 1:] - gray_f[:, :-1]
    # align to intersection region
    mag = np.sqrt(dx[:, 1:]**2 + dy[1:, :]**2)
    return float(mag.mean())

def standard_deviation_SD(gray_f):
    return float(gray_f.std())

def spatial_frequency_SF(gray_f):
    RF = np.sqrt(np.mean((gray_f[1:, :] - gray_f[:-1, :])**2))
    CF = np.sqrt(np.mean((gray_f[:, 1:] - gray_f[:, :-1])**2))
    return float(np.sqrt(RF**2 + CF**2))

def bicubic_resize(img_rgb_u8, size_wh):
    W, H = size_wh
    return cv2.resize(img_rgb_u8, (W, H), interpolation=cv2.INTER_CUBIC)

# ---- collect files ----
hr_paths = sorted(glob.glob(HR_GLOB))
rows = []

for p_hr in hr_paths:
    base = os.path.basename(p_hr).replace('_fused_hr.png', '')
    p_dbg = os.path.join(DBG_DIR, f"{base}_sr_input.png")
    if not os.path.exists(p_dbg):
        print(f"[skip] missing debug SR input for {base}: {p_dbg}")
        continue

    # --- fusion metrics on debug image (LR fused RGB -> grayscale) ---
    dbg_gray_u8 = read_gray(p_dbg)
    dbg_gray_f  = dbg_gray_u8.astype(np.float32)

    EN = entropy_EN(dbg_gray_u8)
    AG = average_gradient_AG(dbg_gray_f)
    SD = standard_deviation_SD(dbg_gray_f)
    SF = spatial_frequency_SF(dbg_gray_f)

    # --- PSNR/SSIM on final HR vs bicubic upsample of debug LR ---
    hr_rgb = to_u8(read_rgb(p_hr))
    H, W = hr_rgb.shape[:2]
    dbg_rgb = to_u8(read_rgb(p_dbg))
    bi_rgb  = bicubic_resize(dbg_rgb, (W, H))

    PS = psnr(bi_rgb, hr_rgb, data_range=255)
    SS = ssim(bi_rgb, hr_rgb, data_range=255, channel_axis=2)

    rows.append({
        "image": base,
        "EN_debug": EN,
        "AG_debug": AG,
        "SD_debug": SD,
        "SF_debug": SF,
        "PSNR_vs_bicubic": PS,
        "SSIM_vs_bicubic": SS,
        "HR_path": p_hr,
        "Debug_path": p_dbg
    })

# ---- save to Excel ----
if rows:
    df = pd.DataFrame(rows)

    # summary stats
    summary = {
        "count": len(df),
        "EN_debug_mean":  df["EN_debug"].mean(),
        "AG_debug_mean":  df["AG_debug"].mean(),
        "SD_debug_mean":  df["SD_debug"].mean(),
        "SF_debug_mean":  df["SF_debug"].mean(),
        "PSNR_mean":      df["PSNR_vs_bicubic"].mean(),
        "SSIM_mean":      df["SSIM_vs_bicubic"].mean(),
    }
    df_summary = pd.DataFrame([summary])

    with pd.ExcelWriter(OUTPUT_XLS, engine="xlsxwriter") as writer:
        df.to_excel(writer, index=False, sheet_name="per_image")
        df_summary.to_excel(writer, index=False, sheet_name="summary")

    print(f"Saved metrics for {len(df)} image(s) to: {OUTPUT_XLS}")
else:
    print("No image pairs found. Make sure OUT_DIR contains '*_fused_hr.png' and 'debug/*_sr_input.png'.")


Saved metrics for 52 image(s) to: /content/drive/MyDrive/Hybrid/output/M3FD_output/M3FD_50-images_metrics.xlsx
